# Multimodal (Vision-Language) — โจทย์แบบ "รูป -> ข้อความ"

เช่น บรรยายรูปเป็นภาษาไทย (image captioning), ตอบคำถามจากรูป (VQA), อ่านตัวอักษรในรูป (OCR)

วิธีในโน้ตบุ๊กนี้ (สำหรับ image captioning ไทย):
- วิธีที่ 1 (ง่ายสุด มือใหม่ทำแค่นี้) = ใช้โมเดลที่เทรนกับ Thai-COCO มาแล้ว generate ได้เลย ไม่ต้องเทรน
- วิธีที่ 2 (ไม่บังคับ) = `BLIP` fine-tune (เบา)
- วิธีที่ 3 (ขั้นสูง ต้อง GPU 16GB) = `PaliGemma2` + LoRA คะแนนดีสุด


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ input มีรูป และ output เป็น "ข้อความ" (ประโยค/คำตอบ)
ถ้า output เป็น "1 คลาส" -> ไปหัวข้อ 01 Computer Vision

สำคัญมาก: metric คือ `BLEU` ที่ต้องตัดคำไทยด้วย `newmm` ก่อน
ใช้ฟังก์ชัน `thai_bleu()` ในโน้ตบุ๊กวัดคะแนนในเครื่องก่อนส่งทุกครั้ง
ต้องแก้ (`# << แก้`): ชื่อ competition, path โฟลเดอร์รูป/ไฟล์ JSON

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install -U transformers accelerate datasets pillow pandas torch sentencepiece
!pip -q install pythainlp nltk          # ใช้วัด Thai BLEU
!pip -q install peft bitsandbytes       # วิธีที่ 3 เท่านั้น

## ขั้นที่ 2 — ตั้งค่า Kaggle แล้วดาวน์โหลดข้อมูล

ต้องมีไฟล์ `kaggle.json` ก่อน วิธีได้มา: เข้า kaggle.com -> กดรูปโปรไฟล์ -> Settings -> Account -> Create New Token
จะได้ไฟล์ `kaggle.json` หน้าตา `{"username":"...","key":"..."}`

- ถ้ารันบน `Kaggle Notebook`: ข้อมูลอยู่ที่ `/kaggle/input/...` แล้ว ไม่ต้องทำอะไร รันผ่านได้เลย
- ถ้ารันบน `Google Colab / เครื่องตัวเอง`: เอา username กับ key ใส่ในเซลล์ข้างล่าง (จุด `# << แก้`)

In [ ]:
import os, glob, subprocess

COMP = "super-ai-engineer-ss-6-thai-language-image-captioning"      # << แก้ตรงนี้: slug ของ competition (คือส่วนท้าย URL หลังคำว่า /competitions/)
KAGGLE_USERNAME = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "peetwan1"
KAGGLE_KEY      = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) คีย์ยาว ๆ จาก kaggle.json

def get_data(comp):
    # ถ้าอยู่บน Kaggle ข้อมูลถูกต่อไว้ให้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle แล้ว ข้อมูลอยู่ที่", kpath); return kpath
    # ถ้าไม่ใช่ Kaggle: เขียน token แล้วโหลดด้วย kaggle CLI
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("ไฟล์ที่ดาวน์โหลดมา (ดูชื่อไฟล์/โฟลเดอร์ แล้วเอาไปแก้ในเซลล์ถัดไป):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — path + ตัววัด Thai BLEU (นี่คือ metric จริง)

In [ ]:
import os, glob, json, pandas as pd, torch
from PIL import Image
TRAIN_IMG  = os.path.join(DATA_DIR,"train")     # << แก้ path ให้ตรง
TEST_IMG   = os.path.join(DATA_DIR,"test")
SAMPLE_SUB = os.path.join(DATA_DIR,"sample_submission.csv")
js = glob.glob(os.path.join(DATA_DIR,"*train*.json")); TRAIN_JSON = js[0] if js else None
sample = pd.read_csv(SAMPLE_SUB); ID_COL, CAP_COL = sample.columns[0], sample.columns[1]
print("รูป test:", TEST_IMG, "| sample คอลัมน์:", list(sample.columns)); display(sample.head())

from pythainlp.tokenize import word_tokenize
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
def th_tok(s): return [w for w in word_tokenize(str(s),engine="newmm",keep_whitespace=False) if w.strip()]
def thai_bleu(list_of_refs, hyps):
    refs=[[th_tok(r) for r in rs] for rs in list_of_refs]; hyp=[th_tok(h) for h in hyps]
    return corpus_bleu(refs,hyp,weights=(0.25,0.25,0.25,0.25),smoothing_function=SmoothingFunction().method4)
def write_submission(pred_dict, out="submission.csv"):
    def look(idv):
        s=str(idv)
        for k in (s, s+".jpg", os.path.basename(s)):
            if k in pred_dict: return pred_dict[k]
        return "ภาพ"   # fallback ไทยสั้น ๆ กัน BLEU=0
    o=sample.copy(); o[CAP_COL]=o[ID_COL].map(look)
    o.to_csv(out,index=False,encoding="utf-8-sig"); print("บันทึก",out,o.shape); return o

## วิธีที่ 1 — โมเดล Thai-COCO สำเร็จรูป (ง่ายสุด ไม่ต้องเทรน)

ใช้โมเดลที่เคยเทรนกับคลังข้อมูลเดียวกัน generate คำบรรยายได้เลยใน <1 ชม.

In [ ]:
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer
M="Natthaphon/thaicapgen-convnext-phayathai"   # << แก้โมเดลได้
dev="cuda" if torch.cuda.is_available() else "cpu"
fe=AutoImageProcessor.from_pretrained(M); tk=AutoTokenizer.from_pretrained(M)
model=AutoModel.from_pretrained(M,trust_remote_code=True).to(dev).eval()
pred={}
for p in sorted(glob.glob(os.path.join(TEST_IMG,"*"))):
    px=fe(images=[Image.open(p).convert("RGB")],return_tensors="pt").pixel_values.to(dev)
    with torch.no_grad(): ids=model.generate(px,max_length=120,num_beams=4)
    pred[os.path.basename(p)]=tk.batch_decode(ids,skip_special_tokens=True)[0].strip()
write_submission(pred,"submission.csv")

## วิธีที่ 2 — BLIP fine-tune (ไม่บังคับ)

โค้ดเต็มอยู่ใน reference_cheatsheet.md (โหลดคู่รูป-คำบรรยายจาก JSON มาเทรนต่อ)
มือใหม่ข้ามได้ ใช้วิธีที่ 1 ก็ได้คะแนนแล้ว

## วิธีที่ 3 — PaliGemma2 + LoRA (ขั้นสูง ต้อง GPU 16GB คะแนนดีสุด)

โค้ดเต็มอยู่ใน reference_cheatsheet.md หัวข้อ D (prompt `caption th` + LoRA 4-bit)

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
FILE = "submission.csv"   # << แก้เป็นชื่อไฟล์ที่คะแนนดีสุด
!kaggle competitions submit -c {COMP} -f {FILE} -m "thai coco pretrained captioning"
!kaggle competitions submissions -c {COMP} | head